# 🎭 Kerem Avatar — Dijital İkiz
**Ses Klonu + Konuşan Avatar** — Linly-Talker tabanlı

> GPU: T4 veya daha iyi seçin (Runtime → Change runtime type → T4 GPU)

In [ ]:
# GPU kontrolü
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ GPU YOK - Runtime > T4 GPU seç')
print('CUDA:', torch.version.cuda)

## 1️⃣ Kurulum

In [ ]:
import os
os.chdir('/content')

# Kerem Avatar reposunu klonla
!git clone https://github.com/Kerem23-elz/Kerem-avatar.git
os.chdir('/content/Kerem-avatar')

# Linly-Talker klonla
!git clone --depth=1 https://github.com/Kedreamix/Linly-Talker.git

In [ ]:
# Temel bağımlılıklar
!pip install -q torch torchvision torchaudio
!pip install -q -r Linly-Talker/requirements_app.txt
!pip install -q gfpgan face_alignment pyyaml soundfile

## 2️⃣ Model İndirme

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

print('SadTalker + Linly-Talker modelleri indiriliyor...')
snapshot_download(
    repo_id='Kedreamix/Linly-Talker',
    local_dir='Linly-Talker/checkpoints',
    ignore_patterns=['*.bin.index.json'],
)
print('✓ Linly-Talker modelleri hazır')

In [ ]:
print('GPT-SoVITS modelleri indiriliyor...')
snapshot_download(
    repo_id='lj1995/GPT-SoVITS',
    local_dir='Linly-Talker/GPT_SoVITS/pretrained_models',
)
print('✓ GPT-SoVITS modelleri hazır')

## 3️⃣ Dosyaları Yükle

In [ ]:
from google.colab import files
import os

os.makedirs('inputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('📹 Ses örneği için VIDEO yükle (MP4):')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
os.rename(video_path, f'inputs/{video_path}')
VIDEO_PATH = f'inputs/{video_path}'
print(f'✓ Video: {VIDEO_PATH}')

In [ ]:
print('🖼️ FOTOĞRAF yükle (JPG/PNG):')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
os.rename(img_path, f'inputs/{img_path}')
IMAGE_PATH = f'inputs/{img_path}'
print(f'✓ Fotoğraf: {IMAGE_PATH}')

## 4️⃣ Ses Klonlama

In [ ]:
import subprocess, sys
sys.path.insert(0, 'Linly-Talker')
sys.path.insert(0, 'Linly-Talker/GPT_SoVITS')

# Videodan ses çıkar
REF_AUDIO = 'inputs/ref_audio.wav'
subprocess.run([
    'ffmpeg', '-i', VIDEO_PATH,
    '-vn', '-acodec', 'pcm_s16le',
    '-ar', '22050', '-ac', '1',
    REF_AUDIO, '-y'
], check=True)
print(f'✓ Ses örneği çıkarıldı: {REF_AUDIO}')

# Ses örneğini dinle
from IPython.display import Audio, display
display(Audio(REF_AUDIO))

In [ ]:
# Avatarın söyleyeceği metni belirle
AVATAR_TEXT = "Merhaba! Ben Kerem'in dijital ikiziyim. Yapay zeka ile oluşturuldum."  # @param {type:"string"}
LANGUAGE = "auto"  # @param ["auto", "tr", "en", "zh"]

print(f'Metin: {AVATAR_TEXT}')

In [ ]:
# GPT-SoVITS ile ses klonla
CLONED_AUDIO = 'outputs/klonlanan_ses.wav'

from GPT_SoVITS.inference_webui import get_tts_wav
import soundfile as sf
import numpy as np

print('Ses klonlanıyor...')
result_gen = get_tts_wav(
    ref_wav_path=REF_AUDIO,
    prompt_text='',
    prompt_language=LANGUAGE,
    text=AVATAR_TEXT,
    text_language=LANGUAGE,
    ref_free=True,
)

chunks, sr = [], None
for sample_rate, chunk in result_gen:
    sr = sample_rate
    chunks.append(chunk)

sf.write(CLONED_AUDIO, np.concatenate(chunks), sr)
print(f'✓ Klonlanan ses: {CLONED_AUDIO}')

display(Audio(CLONED_AUDIO))

## 5️⃣ Konuşan Avatar Oluştur

In [ ]:
AVATAR_MODEL = "sadtalker"  # @param ["sadtalker", "musetalk"]
OUTPUT_VIDEO = 'outputs/dijital_ikiz.mp4'

if AVATAR_MODEL == 'sadtalker':
    from TFG import SadTalker
    
    sadtalker = SadTalker(
        checkpoint_path='Linly-Talker/checkpoints',
        config_path='Linly-Talker/src/config',
        lazy_load=True,
    )
    
    print('Avatar videosu oluşturuluyor (SadTalker)...')
    result = sadtalker.test(
        source_image=IMAGE_PATH,
        driven_audio=CLONED_AUDIO,
        still_mode=False,
        use_enhancer=True,
        enhancer='gfpgan',
        preprocess='crop',
        size=512,
        result_dir='outputs',
    )
    if result:
        import shutil
        shutil.move(result, OUTPUT_VIDEO)

print(f'✓ Dijital İkiz hazır: {OUTPUT_VIDEO}')

## 6️⃣ Sonuç

In [ ]:
from IPython.display import Video, display

print('🎉 Dijital İkiz Videosu:')
display(Video(OUTPUT_VIDEO, width=512))

# İndir
files.download(OUTPUT_VIDEO)
print('✓ Video indirildi!')